In [ ]:
active_region = "west"
table_name = "icetabledemo6"
database = "berg"

In [ ]:
passive_region = "west" if active_region == "east" else "east"
region_name = f"us-{active_region}-2"
active_bucket = f"iceberg-wh-{active_region}"
passive_bucket = f"iceberg-wh-{passive_region}"
active_metadata = f"metadata-{active_region}"
passive_metadata = f"metadata-{passive_region}" 

In [ ]:
from pyspark.sql import SparkSession
from pyspark import SparkConf
import boto3
import subprocess

sp_conf = SparkConf() 
sp_conf.set("spark.sql.catalog.glue_catalog", "org.apache.iceberg.spark.SparkCatalog")
sp_conf.set("spark.sql.catalog.glue_catalog.warehouse", f"s3://{active_bucket}/")
sp_conf.set("spark.sql.catalog.glue_catalog.catalog-impl", "org.apache.iceberg.aws.glue.GlueCatalog")
sp_conf.set("spark.sql.catalog.glue_catalog.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
sp_conf.set("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
sp_conf.set("spark.hadoop.fs.s3a.aws.credentials.provider","com.amazonaws.auth.DefaultAWSCredentialsProviderChain")
spark.sparkContext._jsc.hadoopConfiguration().set("fs.s3a.aws.credentials.provider", "com.amazonaws.auth.DefaultAWSCredentialsProviderChain")

spark = SparkSession.builder \
    .appName("Glue-Iceberg-Integration") \
    .config(conf=sp_conf) \
    .getOrCreate()

In [ ]:
def get_dynamo_latest_metadata_info(d, t):
    dynamodb = boto3.resource('dynamodb', region_name=region_name)
    # Get a table resource
    table = dynamodb.Table('latest_metadata')
    key = {'dbtable': f"{d}.{t}"}
    try:
        response = table.get_item(Key = key)
        return response["Item"]["metadatafile"]
    except Exception as e:
        print("Error getting item:", e)

def set_dynamo_with_new_latest_metadata_info(d, t, latest_metadata):
    # Create a DynamoDB resource
    dynamodb = boto3.resource('dynamodb', region_name=region_name)
    # Get a table resource
    table = dynamodb.Table('latest_metadata')
    # Define the item to be inserted/updated
    item = {
        'dbtable': f'{d}.{t}',
        'metadatafile': latest_metadata,
    }
    try:
        response = table.put_item(
        Item=item
        )
        print("Item put successfully:", response)
    except Exception as e:
        print("Error putting item:", e)


def get_metadata_from_table(d, t):
    glue = boto3.client("glue", region_name = region_name)
    table = glue.get_table(DatabaseName=d, Name=t)
    parameters = table["Table"]["Parameters"]
    full_path_metadata_location = parameters["metadata_location"]
    return full_path_metadata_location.split('/')[-1]

def update_metadata_table(d, t, latest_metadata):
    glue = boto3.client("glue", region_name = region_name)
    table = glue.get_table(DatabaseName=d, Name=t)
    table_input = table["Table"]
    table_input["Parameters"]["metadata_location"] = f"s3://{active_bucket}/{database}.db/{table_name}/metadata/{latest_metadata}"
    
    keys_to_remove = ['CreateTime', 'UpdateTime', 'IsRegisteredWithLakeFormation', 'CatalogId', 'DatabaseName', 'CreatedBy', 'VersionId', 'IsMultiDialectView']
    
    for key in keys_to_remove:
        if key in table_input: del table_input[key]

    print(table_input)
    glue.update_table(
        DatabaseName=d,
        TableInput=table_input
    )
    return

In [ ]:
%load_ext mermaid_magic

In [ ]:
%%mermaid
sequenceDiagram
    SparkJobEast->>Glue Catalog East: Create DB and Table
    SparkJobEast->>DynamoDB: Update latest Metadata.json for DB/Table
    SparkJobEast->>S3-East: RewriteTablePath for metadata and place files in bucket S3-East under the metadata-west prefix.
    DataSyncWest->>S3-West: Move rewritten files from S3 West(metadata-west prefix) to S3 West (metadata)

In [ ]:
s3_copy = f"aws s3 sync s3://{active_bucket}/{database}.db/{table_name}/{active_metadata} s3://{active_bucket}/{database}.db/{table_name}/metadata/"
subprocess.run(f"{s3_copy}", shell=True, capture_output=True, text=True, check=True)

In [ ]:
%%mermaid
sequenceDiagram
    SparkJobEast->>Glue Catalog East: Create DB and Table
    SparkJobEast->>DynamoDB: Update latest Metadata.json for DB/Table
    SparkJobEast->>S3-East: RewriteTablePath for metadata and place files in bucket S3-East under the metadata-west prefix.
    DataSyncWest->>S3-West: Move rewritten files from S3 West(metadata-west prefix) to S3 West (metadata)
    SparkJobWest->>Glue Catalog West: Create DB
    SparkJobWest->>DynamoDB:Get the latest metadata file name from DynamoDB
    SparkJobWest->>Glue Catalog West: Register the table with Glue Catalog West 



In [ ]:
spark.sql(f"""
    CREATE DATABASE IF NOT EXISTS glue_catalog.{database} 
""")

In [ ]:
latest_metadata = get_dynamo_latest_metadata_info(database, table_name) 
spark.sql(f"""
  CALL glue_catalog.system.register_table(
    table => '{database}.{table_name}',
    metadata_file => 's3://{active_bucket}/{database}.db/{table_name}/metadata/{latest_metadata}'
  )
""")

In [ ]:
spark.sql(f"select count(*) from glue_catalog.{database}.{table_name}").show()

In [ ]:
#STOP AND PRODUCE SOME DATA IN THE EAST

In [ ]:
#STOP AND PRODUCE SOME DATA IN THE EAST

In [ ]:
#STOP AND PRODUCE SOME DATA IN THE EAST

In [ ]:
#STOP AND PRODUCE SOME DATA IN THE EAST

In [ ]:
#STOP AND PRODUCE SOME DATA IN THE EAST

In [ ]:
%%mermaid
sequenceDiagram
    SparkJobEast->>Glue Catalog East: Create DB and Table
    SparkJobEast->>DynamoDB: Update latest Metadata.json for DB/Table
    SparkJobEast->>S3-East: RewriteTablePath for metadata and place files in bucket S3-East under the metadata-west prefix.
    DataSyncWest->>S3-West: Move rewritten files from S3 West(metadata-west prefix) to S3 West (metadata)
    SparkJobWest->>Glue Catalog West: Create DB
    SparkJobWest->>DynamoDB:Get the latest metadata file name from DynamoDB
    SparkJobWest->>Glue Catalog West: Register the table with Glue Catalog West
    SparkJobEast->>S3-East: Publish data to east table.
    SparkJobEast->>DynamoDB: Update latest Metadata.json for DB/Table
    SparkJobEast->>S3-East: RewriteTablePath for metadata and place files in bucket S3-East under the metadata-west prefix.
    DataSyncWest->>S3-West: Move rewritten files from S3 West(metadata-west prefix) to S3 West (metadata)
    

In [ ]:
s3_copy = f"aws s3 sync s3://{active_bucket}/{database}.db/{table_name}/{active_metadata} s3://{active_bucket}/{database}.db/{table_name}/metadata/"
subprocess.run(f"{s3_copy}", shell=True, capture_output=True, text=True, check=True)

In [ ]:
%%mermaid
sequenceDiagram
    SparkJobEast->>Glue Catalog East: Create DB and Table
    SparkJobEast->>DynamoDB: Update latest Metadata.json for DB/Table
    SparkJobEast->>S3-East: RewriteTablePath for metadata and place files in bucket S3-East under the metadata-west prefix.
    DataSyncWest->>S3-West: Move rewritten files from S3 West(metadata-west prefix) to S3 West (metadata)
    SparkJobWest->>Glue Catalog West: Create DB
    SparkJobWest->>DynamoDB:Get the latest metadata filename name from DynamoDB
    SparkJobWest->>Glue Catalog West: Register the table with Glue Catalog West
    SparkJobEast->>S3-East: Publish data to east table.
    SparkJobEast->>DynamoDB: Update latest Metadata.json for DB/Table
    SparkJobEast->>S3-East: RewriteTablePath for metadata and place files in bucket S3-East under the metadata-west prefix.
    DataSyncWest->>S3-West: Move rewritten files from S3 West(metadata-west prefix) to S3 West (metadata)
    SparkJobWest->>DynamoDB:Get the latest metadata filename name from DynamoDB
    SparkJobWest->>Glue Catalog West: Update metadata Glue Catalog West with new value of metadata.json

In [ ]:
latest_metadata = get_dynamo_latest_metadata_info(database, table_name) 
update_metadata_table(database,table_name, latest_metadata)

In [ ]:
spark.sql(f"select count(*) from glue_catalog.{database}.{table_name}").show()

In [ ]:
spark.sql(f"SELECT * FROM glue_catalog.{database}.{table_name}.history").show(truncate=False)

In [ ]:
spark.sql(f"SELECT file, latest_snapshot_id FROM glue_catalog.{database}.{table_name}.metadata_log_entries").show(truncate=False)